# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [ ]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder

import polars as pl
from pathlib import Path

In [ ]:
df = pl.DataFrame(

    {

        "a": [1, 2, 3, 4],

        "b": [0.5, 4, 10, 13],

        "c": [True, True, False, True],

    }

)

In [ ]:
df = df.with_columns(pl.lit(None).alias("trial"), pl.lit(None).alias("other_trial"))

In [ ]:
df

In [ ]:
df.with_columns(pl.col("trial").cast(pl.Int64), pl.col("other_trial").cast(pl.Int64))

In [ ]:
df["a"].dtype

In [ ]:
aoi_file = Path("/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/arg_pisacowsmilk_10_aoi.csv")

In [ ]:
aoi_df = pl.read_csv(aoi_file)

In [ ]:
aoi_df["secondary_unit_of_analysis_idx"]

In [ ]:
df = df.with_columns(pl.lit(None).alias("suoa"))

In [ ]:
df["suoa"].sum() == None

In [ ]:
df = df.with_columns(pl.Series(values=["a", "b", "c", "d"]).alias("d"))

In [ ]:
df

In [ ]:
df.drop("d", "suoa")

In [ ]:
df["suoa"].dtype == pl.Null

In [ ]:
df["d"].dtype == pl.Null

In [ ]:
# If you have a specific config file, load it here:
# settings.load_from_yaml("data/MultiplEYE_<...>/multipleye_settings_preprocessing.yaml")

In [ ]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [ ]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Recalculate:     {settings.RECALCULATE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [ ]:
preprocessed_stimulus_path = Path("/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025")

In [ ]:
destination_aoi_path = Path("/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1")

In [ ]:
aoi_files = list(destination_aoi_path.glob("[!.]*.csv"))

In [ ]:
aoi_file = aoi_files[0]
aoi_question_file = aoi_files[1]

In [ ]:
uoa_file = preprocessed_stimulus_path / "custom_units_of_analysis" / aoi_file.name

In [ ]:
aoi_file

In [ ]:
uoa_column_name = "word"
uoa_idx_column_name = "word_idx"

In [ ]:
uoa_data = pl.read_csv(uoa_file)[
            [uoa_column_name, uoa_idx_column_name, "char", "char_idx", "page",]
        ]

In [ ]:
uoa_data

In [ ]:
uoa_data = uoa_data.sort(by=["page", "char_idx"])

In [ ]:
uoa_data

In [ ]:
aoi_data = pl.read_csv(aoi_file)

In [ ]:
aoi_data

In [ ]:
aoi_data.sort(by=["page", "char_idx"])

In [ ]:
uoa_chars = list(uoa_data["char"])

In [ ]:
uoa_chars

In [ ]:
aoi_data = pl.read_csv(aoi_question_file)

In [ ]:
aoi_data = aoi_data.sort(by=["page", "line_idx", "char_idx"])

In [ ]:
aoi_data.filter(aoi_data["question_image_version"] == "question_images_version_1")

In [ ]:
aoi_question_file

In [ ]:
uoa_file

In [ ]:
uoa_data = uoa_data.sort(by=["page", "char_idx"])

In [ ]:
uoa_data

In [ ]:
uoa_chars = list(uoa_data["char"])

In [ ]:
aoi_chars = list(aoi_data["char"].filter(aoi_data["question_image_version"] == "question_images_version_1"))

In [ ]:
aoi_chars

In [ ]:
aoi_chars = list(aoi_data["char"])
short_aoi_chars = aoi_chars[:-4]

In [ ]:
len(uoa_chars) == len(aoi_chars)

In [ ]:
uoa_chars

In [ ]:
aoi_chars

In [ ]:
import difflib

In [ ]:
for line in difflib.unified_diff(aoi_chars, uoa_chars, lineterm=""):
            print(line)

In [ ]:
aoi_chars == uoa_chars

In [ ]:
new_aois = aoi_data.join(uoa_data, on=["page", "char_idx", "char"])

In [ ]:
aoi_data

In [ ]:
aoi_data = aoi_data.rename(
        {
            "unit_of_analysis_idx": "secondary_unit_of_analysis_idx",
            "unit_of_analysis": "secondary_unit_of_analysis",
            "unit_of_analysis_idx_in_line": "secondary_unit_of_analysis_idx_in_line",})

In [ ]:
def _add_custom_uoa(
    stimuli_folder: Path, aoi_file: Path, uoa_column_name, uoa_idx_column_name, custom_file_name: str = ""
) -> None:
    """
    The custom units of analysis can be added as follows for ZH:
    - They are in a directory called "custom_units_of_analysis" in the data folder
    - The aoi files for the questions are named and structured the same as the original question aoi files.
        - They contain an additional segment and segment_idx column
        - As the text on all pages is for all question versions is the same, only question_images_version_1 is annotated
    - The stimuli text aoi files are name like this: Lit_Alchemist_zh_seg_pages.csv. The first part can be replaced
    by the respective stimulus name
        - They contain an additional word and word_idx column

    How to add it:
    - Merge the custom and the original aoi files on the char and char index as well as the page column!
    - check that all chars listed in both files exactly align. Index and char should be exactly the same.
    - ensure the segment contains the actual characters that it annotates, and that the running segment index is correct
    - for the questions add the segments to all question images versions and make sure the characters match. Only the coordinates should be different for the options

    Output:
    The original aoi files should be exactly the same as before also the file name, but they should contain two additional columns:
        - unit_of_analysis
        - unit_of_analysis_idx

    If there is a word column already in there: rename to secondary_unit_of_analysis and word_idx to secondary_unit_of_analysis_idx.
    The characters and character indices should stay exactly the same.

    If any of the characters and segment characters do not match, it will be reported.

    """

    aoi_data = pl.read_csv(aoi_file)
    if "secondary_unit_of_analysis" in aoi_data:
        # If this file was already merged
        # If the costum units of analysis change, the stimulus folder is automatically copied again and merged
        return

    output_path = stimuli_folder / "merge_output" / aoi_file.name
    aoi_data = aoi_data.sort(by=["page", "line_idx", "char_idx"])
    
    
    uoa_file = stimuli_folder / "custom_units_of_analysis" / aoi_file.name

    if not uoa_file.exists():
        #TODO What about files that are just missing e.g. the question files for Greenlandic
        raise FileNotFoundError(
            f"We could not find a matching uoa file for {aoi_file.name}. Please make "
            f"sure that the uoa file is named exactly as the corresponding aoi file and "
            f"that it is in your data folder in a folder called 'custom_units_of_analysis'."
        )

    if "questions" in uoa_file.name:
        uoa_data = pl.read_csv(uoa_file)[
            [
                uoa_column_name,
                uoa_idx_column_name,
                "char",
                "char_idx",
                "page",
                "question_image_version",
            ]
        ]
        uoa_data = uoa_data.filter(
            uoa_data["question_image_version"] == "question_images_version_1"
        )

        uoa_data = uoa_data[
            [uoa_column_name, uoa_idx_column_name, "char", "char_idx", "page"]
        ]

    else:
        uoa_data = pl.read_csv(uoa_file)[
            [uoa_column_name, uoa_idx_column_name, "char", "char_idx", "page"]
        ]

    uoa_data = uoa_data.sort(by=["page", "char_idx"])
    uoa_chars = list(uoa_data["char"])


    if "questions" in uoa_file.name:
        aoi_chars = list(aoi_data["char"].filter(
            aoi_data["question_image_version"] == "question_images_version_1"))
    else:
        aoi_chars = list(aoi_data["char"])


    # check that the char columns match exactly
    if not aoi_chars == uoa_chars:
        print(f"The custom units of analysis file for the corresponding aoi file {aoi_file.name}, contain different characters. Please verify that the stimuli in both files match exactly.")
        print(f"Using default whitespace delimited units of analysis for {aoi_file.name}")
        return

    aoi_data = aoi_data.rename(
        {
            "unit_of_analysis_idx": "secondary_unit_of_analysis_idx",
            "unit_of_analysis": "secondary_unit_of_analysis",
            "unit_of_analysis_idx_in_line": "secondary_unit_of_analysis_idx_in_line",            
        }
    )

    uoa_data = uoa_data.rename(
        {
            uoa_column_name: "unit_of_analysis",
            uoa_idx_column_name: "unit_of_analysis_idx",
        }
    )
    
    
    # join the segment and segment idx on page and char idx
    new_aois = aoi_data.join(uoa_data, on=["page", "char_idx", "char"])


    # get the line index of the segments
    new_aois = new_aois.with_columns(
        pl.col("unit_of_analysis_idx")
        .rank(method="dense")
        .over(["page", "line_idx"])
        .cast(pl.Int64)
        .alias("unit_of_analysis_idx_in_line")
    )

    col_order = [
        "char_idx",
        "char",
        "top_left_x",
        "top_left_y",
        "width",
        "height",
        "char_idx_in_line",
        "line_idx",
        "page",
        "unit_of_analysis_idx",
        "unit_of_analysis_idx_in_line",
        "unit_of_analysis",
        "secondary_unit_of_analysis_idx",
        "secondary_unit_of_analysis_idx_in_line",
        "secondary_unit_of_analysis",
    ]

    new_aois = new_aois[col_order]
    new_aois = new_aois.sort(by=["page", "line_idx", "char_idx"])
    new_aois.write_csv(output_path)

In [ ]:
aoi_files

In [ ]:
"secondary_unit_of_analysis" in aoi_data

In [ ]:
aoi_data.drop(["char_idx", "char", "bleh"])

In [ ]:
aoi_data["char_idx", "char"]

In [ ]:
for aoi_file in aoi_files:
    if "question" in aoi_file.name:
        uoa_column_name = "segment"
        uoa_idx_column_name = "segment_idx"
    else:
        uoa_column_name = "word"
        uoa_idx_column_name = "word_idx"

    _add_custom_uoa(stimuli_folder=preprocessed_stimulus_path, aoi_file=aoi_file, uoa_column_name=uoa_column_name, uoa_idx_column_name=uoa_idx_column_name)

In [ ]:
aoi_data

In [ ]:
uoa_column_name = "segment"
uoa_idx_column_name = "segment_idx"

In [ ]:
uoa_data = uoa_data.rename({uoa_column_name: "unit_of_analysis",
            uoa_idx_column_name: "unit_of_analysis_idx",})

In [ ]:
uoa_data

In [ ]:
new_aois = aoi_data.join(uoa_data, on=["page", "char_idx", "char"])
new_aois

In [ ]:
new_aois = new_aois.with_columns(
    pl.col("unit_of_analysis_idx")
    .rank(method="dense")
    .over(["page", "line_idx"])
    .cast(pl.Int64)
    .alias("unit_of_analysis_idx_in_line")
)

In [ ]:
new_aois

In [ ]:
col_order = [
        "char_idx",
        "char",
        "top_left_x",
        "top_left_y",
        "width",
        "height",
        "char_idx_in_line",
        "line_idx",
        "page",
        "unit_of_analysis",
        "unit_of_analysis_idx",
        "unit_of_analysis_idx_in_line",
        "secondary_unit_of_analysis_idx",
        "secondary_unit_of_analysis_idx_in_line",
        "secondary_unit_of_analysis",
    ]

In [ ]:
new_aois = new_aois[col_order]

In [ ]:
new_aois

In [ ]:
new_aois.columns

In [ ]:
new_aois = new_aois.sort(by=["page", "line_idx", "char_idx"])

In [ ]:
new_aois.write_csv(aoi_file)

In [ ]:
aoi_file

In [ ]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [ ]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

In [ ]:
from preprocessing.utils.stimulus_versions import (
    configured_version_suffixes,
    default_version_suffix,
    stimulus_folder_name,
    version_suffix_for_pid,
)

In [ ]:
multipleye.data_collection_name

In [ ]:
folder_name = stimulus_folder_name(multipleye.data_collection_name, suffix = None)

In [ ]:
preprocessed_stimulus_path = settings.OUTPUT_DIR / folder_name

In [ ]:
destination_aoi_path = (
            preprocessed_stimulus_path
            / f"aoi_stimuli_{multipleye.language.lower()}_{multipleye.country.lower()}_{multipleye.lab_number}"
        )

In [ ]:
destination_aoi_path

In [ ]:
aoi_files = list(destination_aoi_path.glob("*.csv"))

In [ ]:
aoi_files

In [ ]:
test_file = aoi_files[1]

In [ ]:
test_file

In [ ]:
def rename_aoi_columns(aoi_file: Path) -> None:
    """
    Refactor the aoi files for a given data collection to have consistent naming conventions.
    If necessary, add the custom uoa aois for the given language.
    :param aoi_file: Path to the AOI file
    """

    aoi_data = pl.read_csv(aoi_file)

    # rename columns
    try:
        aoi_data = aoi_data.rename(
            {
                "word_idx": "unit_of_analysis_idx",
                "word": "unit_of_analysis",
                "word_idx_in_line": "unit_of_analysis_idx_in_line",
            },
            strict=False
        )
    except pl.exceptions.ColumnNotFoundError:
        return

    # save the refactored aoi files
    aoi_data = aoi_data.sort(by=["page", "char_idx"])
    aoi_data.write_csv(aoi_file)


In [ ]:
for aoi_file in aoi_files:
    rename_aoi_columns(aoi_file)

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [ ]:
preprocessing.run_preflight_check(multipleye)

## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [ ]:
multipleye.convert_edf_to_asc()

Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [ ]:
multipleye.prepare_session_level_information()

In [ ]:
# print an overview on the data collection and the sessions
multipleye

## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [ ]:
# pick only one session as an example to work with in the next steps
sessions = list(multipleye)  # list of Session objects
sess = sessions[0]  # a real Session
sid = sess.sid  # get the session ID (Sid) from the Session
sid

In [ ]:
sess.stimuli[0].text_stimulus

In [ ]:
type(sess)

### Creating Gaze Frame from ASCII File

In [ ]:
gaze = preprocessing.load_trial_level_raw_data(sid, trial_columns=settings.TRIAL_COLS, load_metadata=True)

In [ ]:
# inspect the gaze samples
gaze.samples.head()

In [ ]:
sess.pm_gaze_metadata = gaze._metadata

Eye movements are recorded in screen pixel coordinates, which depend on stimulus size and monitor setup. To compare gaze behavior across participants, screens, or datasets, it is standard to convert pixel positions 
into **degrees of visual angle (dva)**. Next, we compute **gaze velocity**, which allows us to detect saccades and distinguish them from fixations.

Now we save the gaze data and metadata we just extracted from the asc-file.

In [ ]:
sid.raw_data_dir, sid.metadata_dir

### Output directory structure

All preprocessed data is organised by data type under `preprocessed_data/<data_collection_name>/`. Each data type folder contains one subfolder per session:

```
preprocessed_data/<dcn>/
├── raw_data/
│   └── <session_save_name>/
├── fixations/
│   └── <session_save_name>/
├── saccades/
│   └── <session_save_name>/
├── scanpaths/
│   └── <session_save_name>/
├── reading_measures/
│   └── <session_save_name>/
├── sanity_checks/
│   └── <session_save_name>/
├── metadata/
│   └── <session_save_name>/
│       ├── gaze_metadata.json
│       ├── experiment.yaml
│       ├── calibrations.tsv
│       ├── calibrations.feather
│       ├── validations.tsv
│       ├── validations.feather
│       └── <session_idf>_overview.yaml
├── participant_data.csv
├── <dcn>_overview.yaml
└── stimuli_<dcn>/
```

The `Sid` object provides convenient properties to access each path:

In [ ]:
print(f"Raw data:        {sid.raw_data_dir}")
print(f"Metadata:        {sid.metadata_dir}")
print(f"Fixations:       {sid.fixations_dir}")
print(f"Saccades:        {sid.saccades_dir}")
print(f"Scanpaths:       {sid.scanpaths_dir}")
print(f"Reading measures: {sid.reading_measures_dir}")

In order to have the metadata which is extracted by pymovements available to create out session overview, we get this information from pymovements and store it in our session object.

In [ ]:
sess.pm_gaze_metadata = gaze._metadata
sess.calibrations = gaze.calibrations
sess.validations = gaze.validations

## Stage 2a: Detect Events and Compute Their Properties

Eye-tracking data are typically segmented into events, i.e. `fixations` and `saccades`. Fixations represent moments when the eyes remain relatively still, allowing visual information to be processed, while saccades are the rapid movements between fixations that reposition the gaze. Detecting these events and computing their properties, such as `dispersion`, fixation `duration`, saccade `amplitude`, and `peak velocity`, provides the foundation for analyzing visual behavior and understanding how participants explore a stimulus.

### Fixations

We can detect fixations by applying the `I-VT` or the `I-DT` method.

The **I-VT (Velocity-Threshold Identification)** method distinguishes fixation and saccade points based on their point-to-point velocities. Each point is classified as a fixation if its velocity is below the specified threshold. Consecutive fixation points are then merged into a single fixation. A threshold of 20 degrees/second is commonly used as a default maximum value. Read more about [the IVT algorithm in the documentation](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.ivt.html) 

The **I-DT (Dispersion-Threshold Identification)** method finds fixations by grouping consecutive points within a maximum separation (dispersion) threshold and a minimum duration threshold. The algorithm slides a moving window across the data: if the dispersion within the window is below the threshold, the window represents a fixation and is gradually expanded until the dispersion exceeds the threshold.
Read more about [our implementation of the IDT method](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.idt.html).

We use the `I-VT` algorithm with the following key deafault parameters:
- `minimum duration`: 100 ms 
- `velocity threshold`: 20.0

Such properties as `location`, containing the centroid coordinates of each fixation, and `dispersion` will also be calculated.

In [ ]:
gaze = preprocessing.load_trial_level_events_data(
                        gaze,
                        sess.sid,
                        event_type=settings.FIXATION,
                        file_pattern=None,
                    )

In [ ]:
gaze = preprocessing.load_trial_level_events_data(
                        gaze,
                        sess.sid,
                        event_type=settings.SACCADE,
                        file_pattern=None,
                    )

### Saccades

Saccades are rapid eye movements that shift the point of fixation from one location to another. We detect saccades (or micro-saccades) from the velocity sequence of gaze data using the [microsaccades algorithm](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.microsaccades.html#pymovements.events.detection.microsaccades). This algorithm implements a noise-adaptive velocity threshold, meaning that the detection threshold automatically scales with the noise level of the velocity signal. Such properties as `amplitude` and `peak velocity` of the detected saccades will also be calcuated.

The key default parameters are:
- `threshold_factor`: Multiplier used to determine the velocity threshold relative to the noise level of the signal. The default value is 6. A higher factor makes the algorithm more conservative (detects fewer saccades), while a lower factor makes it more sensitive.
- `minimum_duration`: Defines how long a velocity peak must persist to be classified as a saccade. The duration is expressed in the same units as timesteps. If no timesteps are provided, the value refers to the number of samples (default = 6), which corresponds to about 12 ms at a 500 Hz sampling rate. Shorter events are ignored as noise. 

Save our events data.

In [ ]:
str(sess.sid), sess.sid

## Stage 2b: Map Fixations to AOIs

Once we have the fixations, we can map each of them to the AOIs of the stimulus. The resulting scanpath can then be saved. Note that this features is not yet completely finished.

In [ ]:
stimulus = sess.stimuli[0]
stimulus

In [ ]:
stimulus.text_stimulus

In [ ]:
preprocessing.map_fixations_to_aois(gaze, sess.stimuli)

In [ ]:
# The resulting mapping can be stored as a scanpath, which is a sequence of AOIs that were fixated in the order they were fixated.
preprocessing.save_scanpaths(sid, gaze)

In [ ]:
# save metadata again
preprocessing.save_session_metadata(sid, gaze)

## Stage 3: Calculate AOI-based Measures

In this last step, we calculate the aoi-based measures. These are also refered to as reading measures, as they are typically used in reading research. They include measures such as first pass fixation duration (FPF), total fixation count (TFC), regression path duration (RPD), and many more. These measures are calculated based on the fixations that were mapped to the AOIs in the previous step.

### Fixation-based Metrics

As an intermediate step, the fixations are annotated. These annoataions include:
- The run ID. This ID specifies continuous sequences of fixations on the same word. It is used to calculate first pass and second pass measures.
- Whether the fixation is within the first pass or not
- The index of the preceding word and the following word
- If the saccade entering or leaving the fixation is a regression or not
- Whether it is the first fixation on the word or not

This information is necessary to calculate the reading measures in the next step.

In [ ]:
rm_df = preprocessing.calculate_reading_measures(gaze, sess.stimuli)
preprocessing.save_reading_measures(sid, rm_df)

In [ ]:
rm_df.filter(pl.col("trial") == "trial_1").head()

## Stage 4: Comprehension Question Answers
In addition to gaze data, each session contains answers to comprehension questions.
These are extracted from the ASC messages. The answers are matched to the stimulus order using the `question_order_versions.csv` file in the session's logfiles folder.

In [ ]:
answers_csv = sid.answers_dir / f"{sid}_answers.csv"
question_order_csv = (
    sess.session_folder_path / "logfiles" / "question_order_versions.csv"
)
parsed_answers = preprocessing.parse_answers_from_messages(gaze.messages)
source = "asc"

In [ ]:
parsed_answers

In [ ]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

## Final Steps

In the very end, we can create the session and dataset overview and store them as well. In addition, the participant data can be parsed and stored.

For the MultiplEYE data, there is also the option to create a sanity check report.

In [ ]:
multipleye.create_sanity_check_report(
    gaze,
    sess.session_identifier,
    output_dir=settings.OUTPUT_DIR,
    plotting=True,
    recalculate=True,
)

In [ ]:
multipleye.create_session_overview(sess.session_identifier, path=settings.OUTPUT_DIR)
multipleye.create_dataset_overview(path=settings.OUTPUT_DIR)
multipleye.parse_participant_data(settings.OUTPUT_DIR / "participant_data.csv")

In [ ]:
from preprocessing.psychometric_tests.preprocess_psychometric_tests import (
    preprocess_all_sessions,
)

preprocess_all_sessions()